# Netflix and Disney+ Exploratory Analysis

This notebook examines differences between the Netflix and Disney+ catalogs using the cleaned title-level, country-level, and genre-level datasets.

Because Netflix contains substantially more titles than Disney+, platform comparisons will use proportions and fractional measures in addition to raw counts. The initial analysis focuses on:

1. Overall catalog size
2. Movies versus TV shows
3. Release-year and decade distributions
4. International representation
5. Geographic concentration
6. Countries that are less represented in Disney+ relative to Netflix

The analysis identifies differences in catalog composition. It does not measure viewership, demand, subscriber preferences, revenue, or content quality.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
PROCESSED_DATA_DIR = Path("../data/processed")
OUTPUT_TABLE_DIR = Path("../outputs/tables")

OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
titles = pd.read_csv(
    PROCESSED_DATA_DIR / "streaming_titles_clean.csv",
    parse_dates=["date_added"]
)

countries = pd.read_csv(
    PROCESSED_DATA_DIR / "streaming_titles_by_country.csv",
    parse_dates=["date_added"]
)

genres = pd.read_csv(
    PROCESSED_DATA_DIR / "streaming_titles_by_genre.csv",
    parse_dates=["date_added"]
)

In [4]:
print("Title-level dataset:", titles.shape)
print("Country-level dataset:", countries.shape)
print("Genre-level dataset:", genres.shape)

Title-level dataset: (10257, 16)
Country-level dataset: (12597, 11)
Genre-level dataset: (23233, 11)


In [5]:
catalog_size = (
    titles
    .groupby("platform")
    .agg(total_titles=("content_id", "nunique"))
    .reset_index()
)

display(catalog_size)

,platform,total_titles
0,Disney+,1450
1,Netflix,8807


In [6]:
netflix_total = catalog_size.loc[
    catalog_size["platform"] == "Netflix",
    "total_titles"
].iloc[0]

disney_total = catalog_size.loc[
    catalog_size["platform"] == "Disney+",
    "total_titles"
].iloc[0]

catalog_ratio = netflix_total / disney_total

print(
    f"Netflix's catalog is approximately "
    f"{catalog_ratio:.1f} times the size of Disney+'s catalog."
)

Netflix's catalog is approximately 6.1 times the size of Disney+'s catalog.


In [7]:
content_type_summary = (
    titles
    .groupby(["platform", "type"])
    .agg(title_count=("content_id", "nunique"))
    .reset_index()
)

content_type_summary["platform_total"] = (
    content_type_summary
    .groupby("platform")["title_count"]
    .transform("sum")
)

content_type_summary["platform_percentage"] = (
    content_type_summary["title_count"]
    / content_type_summary["platform_total"]
    * 100
).round(2)

display(content_type_summary)

,platform,type,title_count,platform_total,platform_percentage
0,Disney+,Movie,1052,1450,72.55
1,Disney+,TV Show,398,1450,27.45
2,Netflix,Movie,6131,8807,69.62
3,Netflix,TV Show,2676,8807,30.38


In [8]:
release_year_summary = (
    titles
    .groupby("platform")
    .agg(
        earliest_release=("release_year", "min"),
        latest_release=("release_year", "max"),
        median_release_year=("release_year", "median"),
        average_release_year=("release_year", "mean")
    )
    .round(1)
)

display(release_year_summary)

,earliest_release,latest_release,median_release_year,average_release_year
platform,,,,
Disney+,1928,2021,2011.0,2003.1
Netflix,1925,2021,2017.0,2014.2


In [9]:
titles["release_decade"] = (
    (titles["release_year"] // 10) * 10
).astype(str) + "s"

release_decade_summary = (
    titles
    .groupby(["platform", "release_decade"])
    .agg(title_count=("content_id", "nunique"))
    .reset_index()
)

release_decade_summary["platform_total"] = (
    release_decade_summary
    .groupby("platform")["title_count"]
    .transform("sum")
)

release_decade_summary["platform_percentage"] = (
    release_decade_summary["title_count"]
    / release_decade_summary["platform_total"]
    * 100
).round(2)

release_decade_summary["decade_start"] = (
    release_decade_summary["release_decade"]
    .str.replace("s", "", regex=False)
    .astype(int)
)

release_decade_summary = release_decade_summary.sort_values(
    ["platform", "decade_start"]
)

display(release_decade_summary)

,platform,release_decade,title_count,platform_total,platform_percentage,decade_start
0,Disney+,1920s,1,1450,0.07,1920
1,Disney+,1930s,36,1450,2.48,1930
2,Disney+,1940s,41,1450,2.83,1940
3,Disney+,1950s,50,1450,3.45,1950
4,Disney+,1960s,29,1450,2.00,1960
5,Disney+,1970s,36,1450,2.48,1970
6,Disney+,1980s,49,1450,3.38,1980
7,Disney+,1990s,138,1450,9.52,1990
8,Disney+,2000s,277,1450,19.10,2000
9,Disney+,2010s,554,1450,38.21,2010


In [10]:
known_countries = countries[
    countries["country"] != "Unknown"
].copy()

unknown_country_summary = (
    countries[
        countries["country"] == "Unknown"
    ]
    .groupby("platform")
    .agg(unknown_titles=("content_id", "nunique"))
)

display(unknown_country_summary)


,unknown_titles
platform,
Disney+,219
Netflix,838


In [11]:
country_coverage = (
    known_countries
    .groupby("platform")
    .agg(
        countries_represented=("country", "nunique"),
        titles_with_known_country=("content_id", "nunique")
    )
    .reset_index()
)

display(country_coverage)

,platform,countries_represented,titles_with_known_country
0,Disney+,47,1231
1,Netflix,122,7976


In [13]:
country_summary = (
    known_countries
    .groupby(["platform", "country"])
    .agg(
        unique_titles=("content_id", "nunique"),
        full_associations=("title_credit", "sum"),
        fractional_titles=("fractional_credit", "sum")
    )
    .reset_index()
)

country_summary["known_platform_total"] = (
    country_summary
    .groupby("platform")["fractional_titles"]
    .transform("sum")
)

country_summary["country_share_percentage"] = (
    country_summary["fractional_titles"]
    / country_summary["known_platform_total"]
    * 100
).round(2)

top_countries = (
    country_summary
    .sort_values(
        ["platform", "country_share_percentage"],
        ascending=[True, False]
    )
    .groupby("platform")
    .head(10)
)

display(top_countries)


,platform,country,unique_titles,full_associations,fractional_titles,known_platform_total,country_share_percentage
46,Disney+,United States,1184,1184,1082.399242,1231.000000,87.93
45,Disney+,United Kingdom,101,101,54.182576,1231.000000,4.40
7,Disney+,Canada,77,77,36.065909,1231.000000,2.93
12,Disney+,France,22,22,9.357576,1231.000000,0.76
2,Disney+,Australia,23,23,9.174242,1231.000000,0.75
13,Disney+,Germany,9,9,4.083333,1231.000000,0.33
36,Disney+,South Korea,13,13,3.774242,1231.000000,0.31
8,Disney+,China,10,10,3.324242,1231.000000,0.27
20,Disney+,Japan,10,10,3.324242,1231.000000,0.27
17,Disney+,India,5,5,2.900000,1231.000000,0.24


In [14]:
top_five_country_concentration = (
    country_summary
    .sort_values(
        ["platform", "country_share_percentage"],
        ascending=[True, False]
    )
    .groupby("platform")
    .head(5)
    .groupby("platform")
    .agg(
        top_five_country_share=(
            "country_share_percentage",
            "sum"
        )
    )
    .round(2)
)

display(top_five_country_concentration)

,top_five_country_share
platform,
Disney+,96.77
Netflix,66.74


In [15]:
country_comparison = (
    country_summary
    .pivot(
        index="country",
        columns="platform",
        values="country_share_percentage"
    )
    .fillna(0)
    .reset_index()
)

country_comparison = country_comparison.rename(
    columns={
        "Disney+": "disney_share_percentage",
        "Netflix": "netflix_share_percentage"
    }
)

country_comparison["netflix_minus_disney"] = (
    country_comparison["netflix_share_percentage"]
    - country_comparison["disney_share_percentage"]
).round(2)

display(
    country_comparison.sort_values(
        "netflix_minus_disney",
        ascending=False
    ).head(15)
)

display(
    country_comparison.sort_values(
        "netflix_minus_disney"
    ).head(15)
)



platform,country,disney_share_percentage,netflix_share_percentage,netflix_minus_disney
45,India,0.24,12.58,12.34
53,Japan,0.27,3.43,3.16
115,United Kingdom,4.40,7.14,2.74
100,South Korea,0.31,2.65,2.34
102,Spain,0.08,2.24,2.16
36,France,0.76,2.79,2.03
67,Mexico,0.15,1.70,1.55
33,Egypt,0.01,1.39,1.38
111,Turkey,0.00,1.36,1.36
77,Nigeria,0.00,1.24,1.24


platform,country,disney_share_percentage,netflix_share_percentage,netflix_minus_disney
116,United States,87.93,39.98,-47.95
109,Tanzania,0.04,0.00,-0.04
81,Panama,0.03,0.00,-0.03
41,Guatemala,0.04,0.02,-0.02
97,Slovenia,0.02,0.01,-0.01
3,Angola,0.02,0.01,-0.01
14,Botswana,0.02,0.01,-0.01
1,Albania,0.00,0.00,0.00
30,Dominican Republic,0.00,0.00,0.00
31,East Germany,0.00,0.00,0.00


In [16]:
netflix_country_totals = (
    country_summary[
        country_summary["platform"] == "Netflix"
    ][
        ["country", "fractional_titles"]
    ]
    .rename(
        columns={
            "fractional_titles":
            "netflix_fractional_titles"
        }
    )
)

country_comparison = country_comparison.merge(
    netflix_country_totals,
    on="country",
    how="left"
)

country_comparison["netflix_fractional_titles"] = (
    country_comparison["netflix_fractional_titles"]
    .fillna(0)
)

meaningful_country_gaps = (
    country_comparison[
        country_comparison["netflix_fractional_titles"] >= 20
    ]
    .sort_values(
        "netflix_minus_disney",
        ascending=False
    )
)

display(meaningful_country_gaps.head(15))

,country,disney_share_percentage,netflix_share_percentage,netflix_minus_disney,netflix_fractional_titles
45,India,0.24,12.58,12.34,1003.219048
53,Japan,0.27,3.43,3.16,273.167857
115,United Kingdom,4.40,7.14,2.74,569.521429
100,South Korea,0.31,2.65,2.34,211.309524
102,Spain,0.08,2.24,2.16,178.933333
36,France,0.76,2.79,2.03,222.678571
67,Mexico,0.15,1.70,1.55,135.458333
33,Egypt,0.01,1.39,1.38,110.700000
111,Turkey,0.00,1.36,1.36,108.750000
77,Nigeria,0.00,1.24,1.24,98.833333


In [17]:
genre_vocabulary_summary = (
    genres
    .groupby("platform")
    .agg(
        unique_genre_labels=("genre", "nunique")
    )
)

display(genre_vocabulary_summary)

,unique_genre_labels
platform,
Disney+,44
Netflix,42


In [18]:
netflix_genres = set(
    genres.loc[
        genres["platform"] == "Netflix",
        "genre"
    ]
)

disney_genres = set(
    genres.loc[
        genres["platform"] == "Disney+",
        "genre"
    ]
)

shared_genres = sorted(
    netflix_genres.intersection(disney_genres)
)

netflix_only_genres = sorted(
    netflix_genres - disney_genres
)

disney_only_genres = sorted(
    disney_genres - netflix_genres
)

print("Shared labels:", len(shared_genres))
print(shared_genres)

print("\nNetflix-only labels:", len(netflix_only_genres))
print(netflix_only_genres)

print("\nDisney+-only labels:", len(disney_only_genres))
print(disney_only_genres)

Shared labels: 2
['Docuseries', 'Movies']

Netflix-only labels: 40
['Action & Adventure', 'Anime Features', 'Anime Series', 'British TV Shows', 'Children & Family Movies', 'Classic & Cult TV', 'Classic Movies', 'Comedies', 'Crime TV Shows', 'Cult Movies', 'Documentaries', 'Dramas', 'Faith & Spirituality', 'Horror Movies', 'Independent Movies', 'International Movies', 'International TV Shows', "Kids' TV", 'Korean TV Shows', 'LGBTQ Movies', 'Music & Musicals', 'Reality TV', 'Romantic Movies', 'Romantic TV Shows', 'Sci-Fi & Fantasy', 'Science & Nature TV', 'Spanish-Language TV Shows', 'Sports Movies', 'Stand-Up Comedy', 'Stand-Up Comedy & Talk Shows', 'TV Action & Adventure', 'TV Comedies', 'TV Dramas', 'TV Horror', 'TV Mysteries', 'TV Sci-Fi & Fantasy', 'TV Shows', 'TV Thrillers', 'Teen TV Shows', 'Thrillers']

Disney+-only labels: 42
['Action-Adventure', 'Animals & Nature', 'Animation', 'Anime', 'Anthology', 'Biographical', 'Buddy', 'Comedy', 'Coming of Age', 'Concert Film', 'Crime', 'D

In [19]:
catalog_size.to_csv(
    OUTPUT_TABLE_DIR / "catalog_size_summary.csv",
    index=False
)

content_type_summary.to_csv(
    OUTPUT_TABLE_DIR / "content_type_summary.csv",
    index=False
)

release_decade_summary.to_csv(
    OUTPUT_TABLE_DIR / "release_decade_summary.csv",
    index=False
)

country_summary.to_csv(
    OUTPUT_TABLE_DIR / "country_representation_summary.csv",
    index=False
)

country_comparison.to_csv(
    OUTPUT_TABLE_DIR / "country_platform_comparison.csv",
    index=False
)